# GKP Pulse Replay: Trajectory & Wigner Evolution

Load a pulse saved with `save_pulse`, rebuild the system from its stored
metadata, replay the (piecewise-constant) evolution while collecting the state
at every timestep, and compute the Wigner function along the full trajectory.

The summary figure has two rows:

- **Top row** — left: the optimized pulse amplitudes over time; right: the pulse
  spectrum with the hard bandwidth cutoff $f_{\max}$.
- **Bottom row** — five equally time-spaced Wigner snapshots of the trajectory,
  from the initial vacuum (left) to the final prepared state (right).

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import matplotlib.colors as mpl_colors
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
import numpy as np
import jax.numpy as jnp
import jaxquantum as jqt
from jax import lax
from jax.scipy.linalg import expm

from gkp_optimal_control.grape import (
    TimeGrid,
    FourierBand,
    Penalties,
    load_pulse,
)
from gkp_optimal_control.hamiltonians import (
    kerr_cavity_drift,
    kerr_cavity_squeezing_controls,
    cavity_displacement_controls,
)
from gkp_optimal_control.plotting import set_plot_style
from gkp_optimal_control.utils import wigner_trajectory

set_plot_style()

## Load the pulse

`load_pulse` returns the pulse array together with the metadata dict that was
stored alongside it. Everything needed to rebuild the system (Fock dimension,
Kerr rate, time grid, bandwidth, amplitude bound) lives in that metadata, so no
re-optimization is required.

In [2]:
pulse, meta = load_pulse("../../data/gkp_state_prep.npz")

# np.load returns 0-d arrays; cast to plain Python scalars.
T = float(meta["T_us"])
n_steps = int(meta["n_steps"])
n_fock = int(meta["n_fock"])
K = float(meta["K_rad_per_us"])
f_max = float(meta["f_max_MHz"])
eps_max = float(meta["eps_max"])
F = float(meta["fidelity"])
gkp_delta = float(meta["gkp_delta"])

# Reconstruct the lightweight config objects so the original plotting
# snippets work unchanged.
time_grid = TimeGrid(T=T, n_steps=n_steps)
band = FourierBand(f_max=f_max)
penalties = Penalties(eps_max=eps_max)

print(f"pulse shape : {pulse.shape}")
print(f"T           : {T} us   (dt = {time_grid.dt * 1e3:.3f} ns)")
print(f"n_steps     : {n_steps}")
print(f"n_fock      : {n_fock}")
print(f"K           : {K:.4f} rad/us")
print(f"f_max       : {f_max} MHz")
print(f"eps_max     : {eps_max}")
print(f"fidelity    : {F:.5f}")

pulse shape : (2, 500)
T           : 2.0 us   (dt = 4.000 ns)
n_steps     : 500
n_fock      : 100
K           : 0.6283 rad/us
f_max       : 100.0 MHz
eps_max     : 10.0
fidelity    : 0.99499


## Rebuild the system

Single Kerr-nonlinear cavity in the rotating frame with two-photon (squeezing)
and linear (displacement) I/Q drives — the same four control channels used to
generate the pulse:

$$H/\hbar = \tfrac{K}{2}(a^\dagger)^2 a^2
 + \epsilon_I (a^2 + a^{\dagger 2}) + i\,\epsilon_Q (a^2 - a^{\dagger 2})
 + g_I (a + a^\dagger) + i\,g_Q (a - a^\dagger).$$

The control ordering matches `hamiltonians`: squeezing channels first, then
displacement channels.

In [13]:
h_drift = kerr_cavity_drift(n_fock, kerr=K)
h_squeeze = kerr_cavity_squeezing_controls(n_fock)  # 2 controls (a^2, a^dag^2)
h_displace = cavity_displacement_controls(n_fock)  # 2 controls (a, a^dag)
h_all = jnp.concatenate([h_squeeze, h_displace])  # (4, dim, dim): squeeze then displace

# Keep only the controls actually present in the saved pulse
# (2 = squeezing only, 4 = squeezing + displacement).
n_controls = pulse.shape[0]
h_controls = h_all[:n_controls]

all_pulse_labels = [r"$\epsilon_I(t)$", r"$\epsilon_Q(t)$", r"$g_I(t)$", r"$g_Q(t)$"]
all_spec_labels = [
    r"$|\tilde\epsilon_I(\omega)|$",
    r"$|\tilde\epsilon_Q(\omega)|$",
    r"$|\tilde g_I(\omega)|$",
    r"$|\tilde g_Q(\omega)|$",
]
pulse_labels = all_pulse_labels[:n_controls]
spec_labels = all_spec_labels[:n_controls]

# Initial state: vacuum.
psi_init = jnp.asarray(jqt.basis(n_fock, 0).data.reshape(-1), dtype=jnp.complex128)

print(f"controls in pulse: {n_controls}  ->  {pulse_labels}")

controls in pulse: 2  ->  ['$\\epsilon_I(t)$', '$\\epsilon_Q(t)$']


## Replay the evolution, keeping every timestep

The forward model propagates the state with a piecewise-constant exponential
per timestep. Here we use a history-collecting variant of that same `lax.scan`
so we retain $\psi(t_k)$ at every step (the initial state is prepended, giving
`n_steps + 1` frames spanning $t \in [0, T]$).

In [14]:
def evolve_with_history(pulse, dt, psi_0, h_drift, h_controls):
    """Piecewise-constant propagation that returns the state at every step."""

    def step(psi, eps_k):
        H = h_drift + jnp.einsum("c,cij->ij", eps_k, h_controls)
        U = expm(-1j * dt * H)
        psi_next = U @ psi
        return psi_next, psi_next

    psi_f, history = lax.scan(step, psi_0, pulse.T)  # history: (n_steps, dim)
    full = jnp.concatenate([psi_0[None, :], history], 0)  # prepend initial state
    return psi_f, full


psi_final, states = evolve_with_history(
    jnp.asarray(pulse), time_grid.dt, psi_init, h_drift, h_controls
)
states = np.asarray(states)  # (n_steps + 1, dim)
n_frames = states.shape[0]
t_frames = np.linspace(0.0, T, n_frames)  # us

# Sanity check against the stored fidelity is not possible without the target,
# but we can confirm normalization is preserved by the unitary evolution.
print(f"frames       : {n_frames}")
print(f"|psi_final|^2: {float(np.vdot(psi_final, psi_final).real):.6f}")

frames       : 501
|psi_final|^2: 1.000000


## Wigner functions for all timesteps

`wigner_trajectory` takes a batched `Qarray` and computes the Wigner
distribution for every frame on one shared grid, so all snapshots share
identical $q$/$p$ axes.

In [15]:
x_bound = y_bound = 5.0
grid_points = 200

xvec = np.linspace(-x_bound, x_bound, grid_points)
yvec = np.linspace(-y_bound, y_bound, grid_points)
xvec_j = jnp.asarray(xvec)
yvec_j = jnp.asarray(yvec)

# Five equally time-spaced frames; first = initial (t=0), last = final (t=T).
sel = np.linspace(0, n_frames - 1, 5).round().astype(int)
sel_times = t_frames[sel]  # us

wigners = []
for k in sel:
    psi_k = jqt.Qarray.create(states[k].reshape(n_fock, 1)).unit()
    w = np.asarray(jqt.wigner(psi_k, xvec_j, yvec_j))
    wigners.append(np.real(w))

wigners = np.stack(wigners)  # (5, grid_points, grid_points)
print(
    f"computed {len(wigners)} Wigner frames at steps {sel.tolist()} "
    f"on a {grid_points}x{grid_points} grid"
)

computed 5 Wigner frames at steps [0, 125, 250, 375, 500] on a 100x100 grid


## Summary figure

Top-left: pulse amplitudes. Top-right: spectrum with the bandwidth cutoff.
Bottom: five equally time-spaced Wigner snapshots (shared symmetric color scale)
from the initial vacuum to the final prepared state.

In [ ]:
# Shared symmetric color scale across the five snapshots.
wmax = float(np.abs(wigners).max())
wnorm = mpl_colors.TwoSlopeNorm(vmin=-wmax, vcenter=0.0, vmax=wmax)
X, Y = np.meshgrid(xvec, yvec)

times_ns = np.arange(time_grid.n_steps) * time_grid.dt * 1000  # ns

fig = plt.figure(figsize=(20, 9))
outer = GridSpec(2, 1, height_ratios=[1.0, 1.0], hspace=0.32, figure=fig)

# ---- Top row: pulse (left) and spectrum (right) ----
top = GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[0], wspace=0.22)
ax_pulse = fig.add_subplot(top[0])
ax_spec = fig.add_subplot(top[1])

# Pulse amplitudes over time
for c in range(n_controls):
    ax_pulse.plot(times_ns, pulse[c], label=pulse_labels[c])
ax_pulse.axhline(
    +penalties.eps_max, color="gray", linestyle="--", alpha=0.4, label=r"$\pm\epsilon_{\max}$"
)
ax_pulse.axhline(-penalties.eps_max, color="gray", linestyle="--", alpha=0.4)
ax_pulse.set_xlabel("Time (ns)")
ax_pulse.set_ylabel(r"Drive Amplitude (rad/$\mu$s)")
# ax_pulse.set_title('Optimized pulse')
ax_pulse.legend(ncol=2, fontsize=11)

# Spectrum with bandwidth cutoff
freqs = np.fft.fftshift(np.fft.fftfreq(time_grid.n_steps, d=time_grid.dt))
spectra = np.abs(np.fft.fftshift(np.fft.fft(pulse, axis=-1), axes=-1))
for c in range(n_controls):
    ax_spec.plot(freqs, spectra[c], label=spec_labels[c])
ax_spec.axvline(
    +band.f_max,
    color="gray",
    linestyle="--",
    alpha=0.4,
    label=rf"$\pm f_{{\max}} = {band.f_max:g}$ MHz",
)
ax_spec.axvline(-band.f_max, color="gray", linestyle="--", alpha=0.4)
ax_spec.set_xlabel("Freq. (MHz)")
ax_spec.set_ylabel("|FFT|")
ax_spec.set_yscale("log")
# ax_spec.set_title('Pulse spectrum')
ax_spec.legend(ncol=2, fontsize=11)

# ---- Bottom row: five Wigner snapshots ----
bottom = GridSpecFromSubplotSpec(1, 5, subplot_spec=outer[1], wspace=0.12)
wig_axes = [fig.add_subplot(bottom[i]) for i in range(5)]

im = None
for col, ax in enumerate(wig_axes):
    im = ax.pcolormesh(
        X, Y, wigners[col], cmap="RdBu_r", norm=wnorm, shading="gouraud", rasterized=True
    )
    ax.set_aspect("equal")
    ax.set_xlabel("q")
    if col == 0:
        ax.set_ylabel("p")
    else:
        ax.set_yticklabels([])

    k = sel[col]
    t_us = sel_times[col]
    ax.set_title(rf"$t = {t_us:.2f}\,\mu$s")

# One shared colorbar for the trajectory row.
cbar = fig.colorbar(im, ax=wig_axes, fraction=0.046, pad=0.02)
cbar.set_label(r"$W(q, p)$")

# fig.suptitle(rf'GKP state preparation  —  $F = {F:.4f}$,  $\Delta = {gkp_delta:g}$',
# fontsize=18, y=0.98)
fig.savefig("../../figs/optimal_control/full_gkp_state_prep.pdf", bbox_inches="tight")
plt.show()